In [40]:
import os
from dotenv import load_dotenv

load_dotenv()

google_api_key=os.getenv("GOOGLE_API_KEY")



In [48]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain_groq import ChatGroq




llm=ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

agent=create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]

)

In [43]:
config={"configurable":{"thread_id":"test-1"}}

In [50]:
questions=[
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
    "what is 5*5?",
    "what is 6*5?",
    "what is 7*5?",
    "what is 8*5?",
    "what is 9*5?",


]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)

    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

response

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to perform basic arithmetic operations and obtain the results.\n\n## SUMMARY\nThe conversation involved a series of basic arithmetic operations, including addition, multiplication, division, and subtraction. Key results include: 2+2=4, 10*5=50, 100/4=25, 15-7=8, 3*3=9, 4*4=16, 5*5=25, 6*5=30, 7*5=35.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nCalculate the result of 8*5 to complete the current arithmetic operation.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='8349ade1-c08b-4aa5-ab14-47ba46290d41'), AIMessage(content='8 * 5 = 40.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 238, 'total_tokens': 247, 'completion_time': 0.009440424, 'completion_tokens_details': None, 'prompt_time': 0.012193452, 'prompt_tokens_details': None, 'queue_time': 0.054460818, 'total_time': 

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to perform basic arithmetic operations and obtain the results.\n\n## SUMMARY\nThe conversation involved a series of basic arithmetic operations, including addition, multiplication, division, and subtraction. Key results include: 2+2=4, 10*5=50, 100/4=25, 15-7=8, 3*3=9, 4*4=16, 5*5=25, 6*5=30, 7*5=35, 8*5=40, 9*5=45, and the user has asked and received results for 10*5, 100/4, 15-7, 3*3, 4*4, 5*5, and 6*5.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user has requested the result of 6*5, which is 30, so the next step is to await the user's next arithmetic operation request.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='86bac770-dcb4-4535-8541-412fb95bf478'),
  AIMessage(content='6 * 5 = 30.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 315, 'total_tokens': 324, 'completion_

In [46]:
from langchain_core.callbacks import LLMManagerMixin
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_react_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars // 4
    




C:\Users\harsh\AppData\Local\Temp\ipykernel_4316\2918854953.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent=create_react_agent(


In [ ]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities :
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: {tokens} tokens, {len(response['messages'])}messages")
    print(f"{(response['messages'])}")

response

Paris: 28679 tokens, 10messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to find hotels in various cities, including Paris, London, Tokyo, New York, Dubai, and Singapore, and obtain recommendations for luxury, mid-range, and budget-friendly accommodations.\n\n## SUMMARY\nThe conversation involved a series of requests to find hotels in different cities, including Singapore, Paris, and London. Key results include lists of luxury, mid-range, and budget-friendly hotels in these cities, as well as suggestions for online booking platforms and popular areas to stay. The user has also been provided with information on popular activities to do in each city and transportation options.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe next steps are to provide recommendations for hotels in the remaining cities, including New York, Dubai, and Tokyo, and to offer assistance with booking or suggesting more hotels if needed. Addi

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)-> str:
    """mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str,body:str)-> str:
    """mock function to send an email"""
    return f"email sent to {recipient} with subject '{subject}'"



In [26]:
from langgraph.types import Checkpointer
from langchain.chat_models import init_chat_model
llm=init_chat_model("groq:qwen/qwen3-32b")
agent = create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool": False,
                
            }
        )
    ]
)

In [27]:
from langchain.messages import HumanMessage
config = {"configurable": {"thread_id": "test-approve"}}

result=agent.invoke(
    {"messages":HumanMessage(content="send email to john@test.com with subject 'hello' and body 'how are you?'")},
    config=config
)

In [28]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='f270f55a-e004-4e73-aa35-cc3c4c9d0471'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'hello' and body 'how are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are provided here, so I need to call that function with these parameters. No need to use the read_email_tool since the user isn't asking to read an email. Just make sure the parameters match the required fields. Yep, looks good. I'll structure the tool call with the provided details.\n", 'tool_calls': [{'id': '67czjws8w', 'function': {'arguments': '{"body":"how are you?","recipient":"john@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'comp

In [ ]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! approval needed")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                    
                    
                    "type":"edit",
                    "edited_action":{
                    "name":"send_email_tool",
                    "args":{
                        "recipient":"correct@email.com",
                        "subject":"corrected Subject",
                        "body":"this was edited by human before sending"
                    }
                    
                    
                    
                    }
                    
                    
                    
                    }
                ]
            }
        ),
        config=config
    )

print( f"Result : {result['messages'][-1].content}")

Paused! approval needed
Result : The email has been sent to **correct@email.com** with the subject **"corrected Subject"** (modified from your original request). The body was also adjusted to:  
*"this was edited by human before sending"*  

Would you like to send another email or make further adjustments? 😊
